In [16]:
import pandas as pd
import numpy as np
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

In [17]:
csv_file = r"datos_baterias_mp\baterias_clean.csv"
df = pd.read_csv(csv_file)
df_feat = df.copy()

print(df_feat.shape)
display(df_feat.head())

(6847, 15)


,battery_formula,working_ion,average_voltage,capacity_grav,capacity_vol,energy_grav,energy_vol,max_delta_volume,max_voltage_step,stability_charge,stability_discharge,framework_formula,num_steps,last_updated,energy_estimada
0,Cs0-3Ac,Cs,-0.707763,128.499827,404.648831,-90.947418,-286.395455,6.270133,0.0,0.020409,0.535925,Ac,1,2025-09-19 09:41:17.571210+00:00,-90.947418
1,Ca0-3Ac,Ca,-0.034773,463.113885,1586.172049,-16.103762,-55.155628,2.709372,0.0,0.020409,0.057262,Ac,1,2025-09-19 09:42:20.982522+00:00,-16.103762
2,Li0-1CsAcO3,Li,2.147050,64.606066,371.641866,138.712466,797.933730,0.036626,0.0,0.219559,0.300647,CsAcO3,1,2025-09-19 09:43:22.301779+00:00,138.712466
3,Na0-3Ag,Na,-0.008959,454.679804,1215.877630,-4.073349,-10.892707,5.098846,0.0,0.003609,0.045182,Ag,1,2025-09-19 09:42:57.716505+00:00,-4.073349
4,Li0-3Ag,Li,0.210565,624.785871,1984.816229,131.558310,417.933696,2.736089,0.0,0.003609,0.000000,Ag,1,2025-09-19 09:43:24.372752+00:00,131.558310


### 1. Convertir variable framework_formula

In [18]:
test = df_feat["framework_formula"].dropna().iloc[:10]

for formula in test:
    try:
        print(formula, "→", Composition(formula))
    except Exception as e:
        print(formula, "→ ERROR:", e)

Ac → Ac1
Ac → Ac1
CsAcO3 → Cs1 Ac1 O3
Ag → Ag1
Ag → Ag1
Ag → Ag1
Ag → Ag1
Ag → Ag1
Ag → Ag1
AgAsF6 → Ag1 As1 F6


In [19]:
def safe_composition(formula):
    try:
        return Composition(formula)
    except Exception:
        return None

In [20]:
df_feat["composition"] = (
    df_feat["framework_formula"]
    .astype(str)
    .apply(safe_composition)
)

In [21]:
print(
    "Composiciones convertidas:",
    df_feat["composition"].notna().sum()
)

print(
    "Composiciones no convertidas:",
    df_feat["composition"].isna().sum()
)

Composiciones convertidas: 6847
Composiciones no convertidas: 0


### 2. Comprobación de errores

In [22]:
failed = df_feat[
    df_feat["composition"].isna()
]

display(
    failed[
        [
            "framework_formula",
            "working_ion",
            "average_voltage",
            "max_delta_volume"
        ]
    ].head(20)
)

,framework_formula,working_ion,average_voltage,max_delta_volume


In [23]:
df_feat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6847 entries, 0 to 6846
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   battery_formula      6847 non-null   object 
 1   working_ion          6847 non-null   object 
 2   average_voltage      6847 non-null   float64
 3   capacity_grav        6847 non-null   float64
 4   capacity_vol         6847 non-null   float64
 5   energy_grav          6847 non-null   float64
 6   energy_vol           6847 non-null   float64
 7   max_delta_volume     6847 non-null   float64
 8   max_voltage_step     6847 non-null   float64
 9   stability_charge     6847 non-null   float64
 10  stability_discharge  6847 non-null   float64
 11  framework_formula    6847 non-null   object 
 12  num_steps            6847 non-null   int64  
 13  last_updated         6847 non-null   object 
 14  energy_estimada      6847 non-null   float64
 15  composition          6847 non-null   o

### 3. Generación de descriptores químicos

In [24]:
ep = ElementProperty.from_preset(
    "magpie"
)

c:\venvs\ml\lib\site-packages\matminer\utils\data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


In [25]:
df_features = ep.featurize_dataframe(
    df_feat,
    col_id="composition",
    ignore_errors=True
)

ElementProperty:   0%|          | 0/6847 [00:00<?, ?it/s]

ElementProperty: 100%|██████████| 6847/6847 [04:22<00:00, 26.10it/s]


In [26]:
print(
    "Número de columnas:",
    df_features.shape[1]
)

Número de columnas: 148


In [27]:
display(
    df_features.head()
)

,battery_formula,working_ion,average_voltage,capacity_grav,capacity_vol,energy_grav,energy_vol,max_delta_volume,max_voltage_step,stability_charge,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,Cs0-3Ac,Cs,-0.707763,128.499827,404.648831,-90.947418,-286.395455,6.270133,0.0,0.020409,...,0.0,0.0,0.0,0.0,225.0,225.0,0.0,225.0,0.0,225.0
1,Ca0-3Ac,Ca,-0.034773,463.113885,1586.172049,-16.103762,-55.155628,2.709372,0.0,0.020409,...,0.0,0.0,0.0,0.0,225.0,225.0,0.0,225.0,0.0,225.0
2,Li0-1CsAcO3,Li,2.147050,64.606066,371.641866,138.712466,797.933730,0.036626,0.0,0.219559,...,0.0,0.0,0.0,0.0,12.0,229.0,217.0,98.0,103.2,12.0
3,Na0-3Ag,Na,-0.008959,454.679804,1215.877630,-4.073349,-10.892707,5.098846,0.0,0.003609,...,0.0,0.0,0.0,0.0,225.0,225.0,0.0,225.0,0.0,225.0
4,Li0-3Ag,Li,0.210565,624.785871,1984.816229,131.558310,417.933696,2.736089,0.0,0.003609,...,0.0,0.0,0.0,0.0,225.0,225.0,0.0,225.0,0.0,225.0


### 4. Convertir a varibale binaria (dummie) working_ion

In [28]:
ion_dummies = pd.get_dummies(
    df_features["working_ion"],
    prefix="working_ion",
    dtype=int
)

In [29]:
df_features = pd.concat(
    [
        df_features,
        ion_dummies
    ],
    axis=1
)

### 5. Guardado de df_features

In [30]:
df_features.to_csv(
    r"datos_baterias_mp\baterias_features_engineered.csv",
    index=False
)